In [ ]:
import sqlite3
import json
from pathlib import Path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
db_path = "../input_data/chinook.db"
if not Path(db_path).exists():
    raise FileNotFoundError(f"{db_path} not found!")

def extract_schema(db_path=db_path, out_file="chinook_schema.json"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    docs = []
    for (table_name,) in tables:
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        ddl = cursor.fetchone()[0]
        docs.append({"table": table_name, "ddl": ddl})

    conn.close()

    Path(out_file).write_text(json.dumps(docs, indent=2))
    return docs

schema_docs = extract_schema()

In [ ]:
# -----------------------------
# 2. Prepare Few-shot Examples
# -----------------------------
FEW_SHOT_EXAMPLES = [
    {
        "question": "List all album titles by AC/DC.",
        "sql": """SELECT Title FROM Album
                  JOIN Artist ON Album.ArtistId = Artist.ArtistId
                  WHERE Artist.Name = 'AC/DC';"""
    },
    {
        "question": "Which customers are from Brazil?",
        "sql": """SELECT FirstName, LastName FROM Customer
                  WHERE Country = 'Brazil';"""
    },
    {
        "question": "Show the top 5 selling tracks.",
        "sql": """SELECT Track.Name, SUM(InvoiceLine.Quantity) AS TotalSales
                  FROM InvoiceLine
                  JOIN Track ON InvoiceLine.TrackId = Track.TrackId
                  GROUP BY Track.Name
                  ORDER BY TotalSales DESC
                  LIMIT 5;"""
    }
]

In [ ]:
# -----------------------------
# 3. Build VectorStore
# -----------------------------
def build_vectorstore(schema_docs, few_shots):
    # Convert docs into text chunks
    texts = []
    for d in schema_docs:
        texts.append(f"Table {d['table']}:\n{d['ddl']}")

    for ex in few_shots:
        texts.append(f"Q: {ex['question']}\nSQL: {ex['sql']}")

    embeddings = OpenAIEmbeddings()
    vs = Chroma.from_texts(texts, embedding=embeddings, collection_name="chinook")
    return vs

In [ ]:
# -----------------------------
# 4. Build Retrieval + QA Chain
# -----------------------------
def build_chain(vectorstore):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    llm = ChatOpenAI(model="gpt-5-2025-08-07", temperature=0)

    prompt = ChatPromptTemplate.from_template("""
You are a SQL expert. 
Use only the provided schema and examples.
schema & table names are case sensitive.
Return valid SQL for SQLite.
Wrap SQL in <SQL></SQL> tags.

Context:
{context}

User Question: {question}
""")

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True,
    )
    return qa_chain

In [ ]:
# -----------------------------
# 5. Safe Execution
# -----------------------------
def safe_execute(sql_query, db_path="chinook.db"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        q = sql_query.lower()
        if any(bad in q for bad in ["update", "delete", "insert", "drop", "alter", "create"]):
            return "⚠️ Unsafe query blocked"
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        return rows[:10]
    except Exception as e:
        return f"Error: {e}"
    finally:
        conn.close()

In [ ]:
# -----------------------------
# 6. Run POC
# -----------------------------
if __name__ == "__main__":
    schema_docs = extract_schema()
    vectorstore = build_vectorstore(schema_docs, FEW_SHOT_EXAMPLES)
    qa_chain = build_chain(vectorstore)

    question = "List the names of customers who purchased tracks from the 'Rock' genre."
    response = qa_chain({"query": question})

    print("\n--- Generated SQL ---")
    print(response["result"])

    try:
        sql = response["result"].split("<SQL>")[1].split("</SQL>")[0]
        print("\n--- Execution Result ---")
        print(safe_execute(sql))
    except:
        print("SQL not found in response")

    print("\n--- Retrieved Context ---")
    for doc in response["source_documents"]:
        print(doc.page_content[:200], "...\n")